# AbSLang example usage

This notebook walks through the four main workflows:

1. **Build a search index** from a CSV of antibody sequences
2. **Search** that index with a query antibody, to retrieve the most structurally similar hits
3. **Predict CDR RMSD** directly between two sequences
4. **Search the unpaired OAS indexes**, which are split per study because of the size of the database

Every file path below is a placeholder — replace them with your own CSV, index directory, and model checkpoint/config.

Sequences are written as `HC` + `|` + `LC` for `paired` mode, and as a single chain without `|` for `hc` (heavy chain) and `nb` (VHH) modes. Inputs are validated and trimmed to the Fv region automatically (with ANARCI), so full-length sequences are accepted — supplying Fv directly just skips that step.

Every distance AbSLang reports is a Euclidean distance between sequence embeddings, which is the predicted mean CDR RMSD in Ångström. Smaller means more structurally similar.

Generating an index from a csv file with antibody sequences to search through

In [ ]:
from abslang.build_search_index import build_index_from_csv

In [ ]:
build_index_from_csv(
    csv_path="data/your_sequences.csv",                    #path to the csv file with antibody sequences
    id_column="PDB_file",                                  # Name of the column with the sequence IDs (e.g. antibody ID) that will be returned by search
    seq_column="Sequence",                                 # Name of the column holding the antibody amino acid sequences, format 'HC' + '|' + 'LC' for paired sequences. (Fv is expected; longer sequences are trimmed automatically)
    out_dir="output/your_index_name",                      # The directory that the search index and embeddings will be saved into
    index_type="flat",                                     # 'flat', 'pq', or 'ivfpq' #Use flat for accuracy
    tm_checkpoint_path="models/paired_checkpoint_cpu.ckpt", # Path to transformer model checkpoint
    tm_config_path="models/params.json"                     # Path to transformer model config
)
#Writes into out_dir: index.<index_type> (here index.flat, or index.pq / index.ivfpq),
#seq_list.json (the embedded sequences, in index order) and artifacts.json (a record of the build settings).
#Remember the id_column and seq_column used here: they must be passed to FaissSearcher below.

Searching through a sequence database using a query antibody sequence, to retrieve top-K most structurally similar antibodies

In [ ]:
from abslang.search_antibody_index import FaissSearcher

In [ ]:
searcher = FaissSearcher(
    faiss_index_file="output/your_index_name/index.flat", #path to the search index file written by build_index_from_csv, named index.<index_type>
    csv_file="data/your_sequences.csv", #path to the csv file with antibody sequences, used to create the search index, containing the antibody amino acid sequences
    id_column="PDB_file", #must match the id_column used to build the index; this is what search returns as 'id'
    seq_column="Sequence", #must match the seq_column used to build the index (defaults are 'id' and 'sequence_alignment_aa', e.g. for OAS-formatted csv files)
    device="cpu", #cuda or cpu, device used to embed the input sequence
    mode="paired", #paired for heavy+light antibody pairs, hc for heavy-only sequences, and nb for VHH
    tm_checkpoint_path="models/paired_checkpoint_cpu.ckpt",  # checkpoint for the transformer model
    tm_config_path="models/params.json"                      # configuration for the transformer model
)


In [ ]:
import pandas as pd

query_sequence = "QVQLVESGGNVVQPGRSLRLSCTASGFTFSSYGMHWVRQAPDKGLEWVAIIWYDGGNKFYADSVKGRFTISRDNSKDTLYLQMNSLRAEDTAVYYCAKAWYKIDDKYSMDVWGQGTTVTVSS|QSALTQPRSVSGSPGQSVTISCTGTSSDVGGYNYVSWYQQHPGKAPKLMIYDVSKRPSGVPDRFSGSKSGNTASLTISGLQAEDEADYYCCSYAGSYTYVFGTGTKVTVL"
#query sequence in the format 'HC' + '|' + 'LC'. (Fv is expected; longer sequences are trimmed automatically)
results = searcher.search(query_sequence, k=10) #search for the query sequence, specifying the number of results to return
#each hit is a dict with keys 'id' (taken from id_column), 'sequence' and 'distance'
#'distance' is the Euclidean distance = predicted mean CDR RMSD in Angstrom, sorted ascending, so the closest structural matches come first

results_table = pd.DataFrame(results)
results_table #results table of the top most similar antibodies, as a pandas dataframe


Predicting the average CDR RMSD between two antibody sequences. The same embeddings that back the search index can also be compared directly to each other: the Euclidean distance between the embeddings of two sequences is their predicted mean CDR RMSD. No index is needed for this, only the two sequences and the model checkpoint.

In [ ]:
from abslang.large_rmsd_inference import predict_cdr_rmsd

In [ ]:
seq1 = 'QVQLVESGGNVVQPGRSLRLSCTASGFTFSSYGMHWVRQAPDKGLEWVAIIWYDGGNKFYADSVKGRFTISRDNSKDTLYLQMNSLRAEDTAVYYCAKAWYKIDDKYSMDVWGQGTTVTVSS|NVLTQSPAIMSASPGEKVTITCSASSSVSYMHWFQQKPGTSPKLWIYSTSNLASGVPARFSGSGSGTSYSLTISRMEAEDAATYYCQQRSSYPLTFGAGTKLELK'
seq2 = "QVQLQQWGAGLLKPSETLSLTCAVYGGSFSGSYWSWIRQPPGKGLEWIGEVNHSGSTNYNPSLKSRVTISVDTSKNHFSLKLSSVTAADTAVYYCSSVHLRFLEWIDDWGQGTLVTVSS|DIQMTQSPSSLSASVGDRVTITCRASKGIRNDLGWYQQKPGTAPKRLIYAASNLQSGVPSRFSGSGSGTEFTLTISSLQPEDFATYYCLQHNSYPQTFGQGTKVEIK"
#Define the two sequences, also in the format 'HC' + '|' + 'LC'. (Fv is expected; longer sequences are trimmed automatically)
predict_cdr_rmsd(seq1, seq2, device='cpu',
                tm_checkpoint_path="models/paired_checkpoint_cpu.ckpt",
                tm_config_path="models/params.json" )
#returns a single float: the predicted mean CDR RMSD between the two sequences, in Angstrom

Searching through the unpaired OAS database. It is too large for a single index, so it is stored as one FAISS index plus a metadata json file per study. `search_heavy_oas` takes the directory holding those pairs, searches each index in turn, then merges and re-ranks the candidates into a single global top-K.

In [ ]:
import pandas as pd
from abslang import search_heavy_oas

In [ ]:
hits = search_heavy_oas(
    directory="Indexes/webapp_heavy_by_study", #directory containing all the faiss indexes and metadata json files
    query="QVQLVESGGNVVQPGRSLRLSCTASGFTFSSYGMHWVRQAPDKGLEWVAIIWYDGGNKFYADSVKGRFTISRDNSKDTLYLQMNSLRAEDTAVYYCAKAWYKIDDKYSMDVWGQGTTVTVSS",
    mode="hc",                          #a single heavy chain, with no '|' separator (this example runs past the Fv, and is trimmed automatically)
    top_k=10, #number of results to return
    per_index_k=None, #The number of sequences to retrieve from each index before merging and re-ranking. If None, it uses top_k.                  
    device="cpu",                       
    tm_checkpoint_path="models/hc_checkpoint.ckpt",
    tm_config_path="models/hc_config.json",
)
#each hit is a dict with keys 'study', 'distance', 'id', 'sequence', 'source_csv', 'row_idx' and 'packed_id',
#sorted by ascending distance (the predicted mean CDR RMSD in Angstrom). 'source_csv' and 'row_idx' locate the hit in the original per-study csv.
pd.DataFrame(hits)